In [1]:
# ============================================================
# Property Sales - Curated Publication
# ============================================================
#
# Input:
#     Cleaned.property_sales_data
#
# Outputs:
#     Curated.property_sales_observation
#         One accepted cleaned row per scrape observation.
#
#     Curated.property_sales_analysis
#         One selected best analytical row per ListingId.
#
# Responsibilities:
#     1. Preserve the complete cleaned observation history.
#     2. Calculate listing-level observation statistics.
#     3. Deterministically select the best observation per listing.
#     4. Add stable analytical bands and sort attributes.
#     5. Publish explainable selection metadata.
#     6. Validate grain and row-count invariants.
#
# This notebook does not:
#     - Resolve multiple ListingId values to one physical property.
#     - Assert complete market coverage.
#     - Build dimensional fact and dimension tables.
#
# Those operations belong in the Gold publication notebook.
# ============================================================

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 3, Finished, Available, Finished, False)

In [2]:
# ============================================================
# 1. Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DecimalType,
    IntegerType,
    LongType,
    StringType,
    TimestampType,
)
from pyspark.sql.window import Window

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 4, Finished, Available, Finished, False)

In [3]:
# ============================================================
# 2. Configuration
# ============================================================

SOURCE_TABLE = "Cleaned.property_sales_data"

TARGET_SCHEMA = "Curated"

OBSERVATION_TABLE_NAME = "property_sales_observation"
OBSERVATION_TABLE = (
    f"{TARGET_SCHEMA}.{OBSERVATION_TABLE_NAME}"
)

ANALYSIS_TABLE_NAME = "property_sales_analysis"
ANALYSIS_TABLE = (
    f"{TARGET_SCHEMA}.{ANALYSIS_TABLE_NAME}"
)

# Selection ranking weights.
#
# These are deliberately separated from the expressions below so the
# ranking policy remains visible and maintainable.
WEIGHT_VALID_NUMERIC_PRICE = 100
WEIGHT_VALID_SOLD_DATE = 80
WEIGHT_USABLE_ADDRESS = 40
WEIGHT_VALID_LAND_SIZE = 30
WEIGHT_PROPERTY_TYPE = 10
WEIGHT_BEDROOMS = 5
WEIGHT_BATHROOMS = 5
WEIGHT_CAR_SPACES = 5
WEIGHT_VALID_SCRAPE_TIMESTAMP = 5

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 5, Finished, Available, Finished, False)

In [4]:
# ============================================================
# 3. Spark configuration
# ============================================================

spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set(
    "spark.sql.legacy.timeParserPolicy",
    "CORRECTED",
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}"
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
# ============================================================
# 4. Helper functions
# ============================================================

def table_exists(table_name):
    """
    Return True when a Spark catalog table exists.
    """
    return spark.catalog.tableExists(table_name)


def boolean_score(condition, weight):
    """
    Return the supplied integer weight when a condition is true.
    """
    return (
        F.when(condition, F.lit(weight))
        .otherwise(F.lit(0))
    )

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 7, Finished, Available, Finished, False)

In [6]:
# ============================================================
# 5. Validate source table
# ============================================================

if not table_exists(SOURCE_TABLE):
    raise RuntimeError(
        f"Required source table does not exist: {SOURCE_TABLE}"
    )

source = spark.table(SOURCE_TABLE)

required_source_columns = [
    # Identity
    "ObservationKey",
    "RecordHash",
    "ListingId",

    # Provenance
    "SourceFile",
    "SourceUrl",
    "SourcePage",
    "SourceOrdinal",
    "RawLoadedAt",
    "RawLoadDate",
    "CleanedAt",

    # URLs
    "DetailPath",
    "DetailUrl",

    # Geography
    "FullAddress",
    "Suburb",
    "Postcode",
    "IsAddressSuppressed",

    # Property attributes
    "PropertyType",
    "Bedrooms",
    "Bathrooms",
    "CarSpaces",
    "LandSizeSqm",

    # Price attributes
    "PriceText",
    "PriceValue",
    "PriceType",
    "HasNumericPrice",
    "IsContactAgent",
    "IsPriceRange",
    "IsPriceFrom",
    "IsPriceWithheld",
    "PricePerSqm",

    # Dates
    "SoldDate",
    "ScrapedAt",
    "ScrapeDate",

    # Repair metadata
    "InputWasColumnShifted",

    # Quality classifications
    "ListingIdQuality",
    "SoldDateQuality",
    "PriceQuality",
    "AddressQuality",
    "LandSizeQuality",
    "ScrapeTimestampQuality",

    # Quality flags
    "HasValidListingId",
    "HasValidSoldDate",
    "HasValidNumericPrice",
    "HasUsableAddress",
    "HasValidLandSize",
]

missing_source_columns = [
    column_name
    for column_name in required_source_columns
    if column_name not in source.columns
]

if missing_source_columns:
    raise RuntimeError(
        "Cleaned source table is missing required columns: "
        + ", ".join(missing_source_columns)
    )


source_summary = source.select(
    F.count("*").alias("SourceRowCount"),
    F.countDistinct("ObservationKey").alias(
        "DistinctObservationCount"
    ),
    F.countDistinct("ListingId").alias(
        "DistinctListingCount"
    ),
    F.sum(
        F.when(
            F.col("ObservationKey").isNull(),
            1,
        ).otherwise(0)
    ).alias("NullObservationKeyCount"),
    F.sum(
        F.when(
            F.col("ListingId").isNull(),
            1,
        ).otherwise(0)
    ).alias("NullListingIdCount"),
).first()

source_row_count = source_summary["SourceRowCount"]
source_listing_count = source_summary[
    "DistinctListingCount"
]

print(f"Source table: {SOURCE_TABLE}")
print(f"Source observations: {source_row_count:,}")
print(
    "Distinct listings: "
    f"{source_listing_count:,}"
)

if source_summary["NullObservationKeyCount"] != 0:
    raise RuntimeError(
        "Cleaned source contains NULL ObservationKey values."
    )

if (
    source_summary["SourceRowCount"]
    != source_summary["DistinctObservationCount"]
):
    raise RuntimeError(
        "Cleaned source contains duplicate ObservationKey values."
    )

if source_summary["NullListingIdCount"] != 0:
    raise RuntimeError(
        "Cleaned source contains NULL ListingId values. "
        "The Cleaned notebook previously reported zero missing IDs."
    )

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 8, Finished, Available, Finished, False)

Source table: Cleaned.property_sales_data
Source observations: 347,887
Distinct listings: 296,422


In [7]:
# ============================================================
# 6. Create curated observation history
# ============================================================

# The Curated observation table preserves the complete valid listing
# observation history while presenting only useful source, parsed and
# quality metadata.
#
# No listing-level deduplication occurs in this output.
observation = (
    source

    # Stable sold-date attributes
    .withColumn(
        "SoldYear",
        F.year("SoldDate").cast(IntegerType()),
    )
    .withColumn(
        "SoldMonth",
        F.month("SoldDate").cast(IntegerType()),
    )
    .withColumn(
        "SoldQuarter",
        F.quarter("SoldDate").cast(IntegerType()),
    )
    .withColumn(
        "SoldYearMonth",
        F.when(
            F.col("SoldDate").isNotNull(),
            F.date_format("SoldDate", "yyyy-MM"),
        ).otherwise(F.lit(None).cast(StringType())),
    )

    # Curated publication metadata
    .withColumn(
        "CuratedAt",
        F.current_timestamp(),
    )

    .select(
        # Observation and listing identity
        "ObservationKey",
        "RecordHash",
        "ListingId",

        # Source provenance
        "SourceFile",
        "SourceUrl",
        "SourcePage",
        "SourceOrdinal",
        "RawLoadedAt",
        "RawLoadDate",
        "CleanedAt",
        "CuratedAt",

        # Listing URLs
        "DetailPath",
        "DetailUrl",

        # Dates
        "SoldDate",
        "SoldYear",
        "SoldMonth",
        "SoldQuarter",
        "SoldYearMonth",
        "ScrapedAt",
        "ScrapeDate",

        # Geography
        "FullAddress",
        "Suburb",
        "Postcode",
        "IsAddressSuppressed",

        # Property attributes
        "PropertyType",
        "Bedrooms",
        "Bathrooms",
        "CarSpaces",
        "LandSizeSqm",

        # Price attributes
        "PriceText",
        "PriceValue",
        "PriceType",
        "HasNumericPrice",
        "IsContactAgent",
        "IsPriceRange",
        "IsPriceFrom",
        "IsPriceWithheld",
        "PricePerSqm",

        # Repair metadata
        "InputWasColumnShifted",

        # Quality classifications
        "ListingIdQuality",
        "SoldDateQuality",
        "PriceQuality",
        "AddressQuality",
        "LandSizeQuality",
        "ScrapeTimestampQuality",

        # Quality flags
        "HasValidListingId",
        "HasValidSoldDate",
        "HasValidNumericPrice",
        "HasUsableAddress",
        "HasValidLandSize",
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 9, Finished, Available, Finished, False)

In [8]:
# ============================================================
# 7. Validate observation-history grain
# ============================================================

observation_validation = observation.select(
    F.count("*").alias("RowCount"),
    F.countDistinct("ObservationKey").alias(
        "DistinctObservationCount"
    ),
    F.countDistinct("ListingId").alias(
        "DistinctListingCount"
    ),
).first()

if observation_validation["RowCount"] != source_row_count:
    raise RuntimeError(
        "Curated observation row count does not match Cleaned. "
        f"Cleaned={source_row_count:,}; "
        f"Curated={observation_validation['RowCount']:,}"
    )

if (
    observation_validation["RowCount"]
    != observation_validation["DistinctObservationCount"]
):
    raise RuntimeError(
        "Curated observation table is not unique by "
        "ObservationKey."
    )

if (
    observation_validation["DistinctListingCount"]
    != source_listing_count
):
    raise RuntimeError(
        "Curated observation distinct listing count differs "
        "from Cleaned."
    )

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 10, Finished, Available, Finished, False)

In [9]:
# ============================================================
# 8. Publish observation-history table
# ============================================================

(
    observation.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OBSERVATION_TABLE)
)

print(
    f"Wrote observation-history table: {OBSERVATION_TABLE}"
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 11, Finished, Available, Finished, False)

Wrote observation-history table: Curated.property_sales_observation


In [10]:
# ============================================================
# 9. Calculate listing-level observation statistics
# ============================================================

listing_statistics = (
    observation
    .groupBy("ListingId")
    .agg(
        F.count("*").cast(IntegerType()).alias(
            "ObservationCount"
        ),
        F.countDistinct("RecordHash").cast(
            IntegerType()
        ).alias(
            "DistinctRecordVersionCount"
        ),
        F.min("ScrapedAt").alias(
            "FirstScrapedAt"
        ),
        F.max("ScrapedAt").alias(
            "LastScrapedAt"
        ),
        F.min("ScrapeDate").alias(
            "FirstScrapeDate"
        ),
        F.max("ScrapeDate").alias(
            "LastScrapeDate"
        ),
        F.countDistinct("PriceValue").cast(
            IntegerType()
        ).alias(
            "DistinctNumericPriceCount"
        ),
        F.countDistinct("FullAddress").cast(
            IntegerType()
        ).alias(
            "DistinctAddressCount"
        ),
        F.countDistinct("SoldDate").cast(
            IntegerType()
        ).alias(
            "DistinctSoldDateCount"
        ),
        F.countDistinct("PropertyType").cast(
            IntegerType()
        ).alias(
            "DistinctPropertyTypeCount"
        ),
        F.max(
            F.col("HasValidNumericPrice").cast("int")
        ).cast(BooleanType()).alias(
            "AnyObservationHasValidNumericPrice"
        ),
        F.max(
            F.col("HasValidSoldDate").cast("int")
        ).cast(BooleanType()).alias(
            "AnyObservationHasValidSoldDate"
        ),
        F.max(
            F.col("HasUsableAddress").cast("int")
        ).cast(BooleanType()).alias(
            "AnyObservationHasUsableAddress"
        ),
        F.max(
            F.col("HasValidLandSize").cast("int")
        ).cast(BooleanType()).alias(
            "AnyObservationHasValidLandSize"
        ),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 12, Finished, Available, Finished, False)

In [11]:
# ============================================================
# 10. Calculate observation selection score
# ============================================================

ranked = (
    observation

    # Numerical score for readability and diagnostics.
    .withColumn(
        "SelectionScore",
        (
            boolean_score(
                F.col("HasValidNumericPrice"),
                WEIGHT_VALID_NUMERIC_PRICE,
            )
            + boolean_score(
                F.col("HasValidSoldDate"),
                WEIGHT_VALID_SOLD_DATE,
            )
            + boolean_score(
                F.col("HasUsableAddress"),
                WEIGHT_USABLE_ADDRESS,
            )
            + boolean_score(
                F.col("HasValidLandSize"),
                WEIGHT_VALID_LAND_SIZE,
            )
            + boolean_score(
                F.col("PropertyType").isNotNull(),
                WEIGHT_PROPERTY_TYPE,
            )
            + boolean_score(
                F.col("Bedrooms").isNotNull(),
                WEIGHT_BEDROOMS,
            )
            + boolean_score(
                F.col("Bathrooms").isNotNull(),
                WEIGHT_BATHROOMS,
            )
            + boolean_score(
                F.col("CarSpaces").isNotNull(),
                WEIGHT_CAR_SPACES,
            )
            + boolean_score(
                F.col("ScrapeTimestampQuality") == "VALID",
                WEIGHT_VALID_SCRAPE_TIMESTAMP,
            )
        ).cast(IntegerType()),
    )

    # Human-readable description of why the record ranks well.
    .withColumn(
        "SelectionAttributes",
        F.concat_ws(
            "|",
            F.when(
                F.col("HasValidNumericPrice"),
                F.lit("VALID_PRICE"),
            ),
            F.when(
                F.col("HasValidSoldDate"),
                F.lit("VALID_SOLD_DATE"),
            ),
            F.when(
                F.col("HasUsableAddress"),
                F.lit("USABLE_ADDRESS"),
            ),
            F.when(
                F.col("HasValidLandSize"),
                F.lit("VALID_LAND_SIZE"),
            ),
            F.when(
                F.col("PropertyType").isNotNull(),
                F.lit("PROPERTY_TYPE"),
            ),
            F.when(
                F.col("Bedrooms").isNotNull(),
                F.lit("BEDROOMS"),
            ),
            F.when(
                F.col("Bathrooms").isNotNull(),
                F.lit("BATHROOMS"),
            ),
            F.when(
                F.col("CarSpaces").isNotNull(),
                F.lit("CAR_SPACES"),
            ),
        ),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 13, Finished, Available, Finished, False)

In [12]:
# ============================================================
# 11. Select best observation per ListingId
# ============================================================

# The order is explicit rather than relying only on SelectionScore.
#
# This ensures the relative importance of each field remains visible
# and stable if weights are adjusted later.
selection_window = (
    Window
    .partitionBy("ListingId")
    .orderBy(
        # Prefer analytically reliable price information.
        F.col("HasValidNumericPrice").desc(),
        F.col("HasNumericPrice").desc(),

        # Prefer reliable sale dates.
        F.col("HasValidSoldDate").desc(),
        F.col("SoldDate").isNotNull().desc(),

        # Prefer usable identity and property attributes.
        F.col("HasUsableAddress").desc(),
        F.col("HasValidLandSize").desc(),
        F.col("PropertyType").isNotNull().desc(),
        F.col("Bedrooms").isNotNull().desc(),
        F.col("Bathrooms").isNotNull().desc(),
        F.col("CarSpaces").isNotNull().desc(),

        # Use the score as a final quality summary.
        F.col("SelectionScore").desc(),

        # When completeness is otherwise equal, prefer the latest
        # observation because it may include later disclosure.
        F.col("ScrapedAt").desc_nulls_last(),
        F.col("ScrapeDate").desc_nulls_last(),

        # Deterministic final tie-breakers.
        F.col("RecordHash").desc(),
        F.col("ObservationKey").desc(),
    )
)

selected = (
    ranked
    .withColumn(
        "SelectionRank",
        F.row_number().over(selection_window),
    )
    .filter(F.col("SelectionRank") == 1)
    .join(
        listing_statistics,
        on="ListingId",
        how="left",
    )
    .withColumnRenamed(
        "ObservationKey",
        "SelectedObservationKey",
    )
    .withColumnRenamed(
        "RecordHash",
        "SelectedRecordHash",
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 14, Finished, Available, Finished, False)

In [13]:
# ============================================================
# 12. Add observation-change classifications
# ============================================================

selected = (
    selected
    .withColumn(
        "HasMultipleObservations",
        (
            F.col("ObservationCount") > 1
        ).cast(BooleanType()),
    )
    .withColumn(
        "HasMultipleRecordVersions",
        (
            F.col("DistinctRecordVersionCount") > 1
        ).cast(BooleanType()),
    )
    .withColumn(
        "HasPriceVariation",
        (
            F.col("DistinctNumericPriceCount") > 1
        ).cast(BooleanType()),
    )
    .withColumn(
        "HasAddressVariation",
        (
            F.col("DistinctAddressCount") > 1
        ).cast(BooleanType()),
    )
    .withColumn(
        "HasSoldDateVariation",
        (
            F.col("DistinctSoldDateCount") > 1
        ).cast(BooleanType()),
    )
    .withColumn(
        "HasPropertyTypeVariation",
        (
            F.col("DistinctPropertyTypeCount") > 1
        ).cast(BooleanType()),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 15, Finished, Available, Finished, False)

In [14]:
# ============================================================
# 13. Add date attributes
# ============================================================

selected = (
    selected
    .withColumn(
        "SoldYear",
        F.year("SoldDate").cast(IntegerType()),
    )
    .withColumn(
        "SoldMonth",
        F.month("SoldDate").cast(IntegerType()),
    )
    .withColumn(
        "SoldMonthName",
        F.when(
            F.col("SoldDate").isNotNull(),
            F.date_format("SoldDate", "MMMM"),
        ).otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "SoldMonthShortName",
        F.when(
            F.col("SoldDate").isNotNull(),
            F.date_format("SoldDate", "MMM"),
        ).otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "SoldQuarter",
        F.quarter("SoldDate").cast(IntegerType()),
    )
    .withColumn(
        "SoldQuarterLabel",
        F.when(
            F.col("SoldDate").isNotNull(),
            F.concat(
                F.lit("Q"),
                F.quarter("SoldDate"),
            ),
        ).otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "SoldYearMonth",
        F.when(
            F.col("SoldDate").isNotNull(),
            F.date_format("SoldDate", "yyyy-MM"),
        ).otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "SoldYearMonthSort",
        F.when(
            F.col("SoldDate").isNotNull(),
            (
                F.year("SoldDate") * 100
                + F.month("SoldDate")
            ).cast(IntegerType()),
        ).otherwise(F.lit(None).cast(IntegerType())),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 16, Finished, Available, Finished, False)

In [15]:
# ============================================================
# 14. Add bedroom bands
# ============================================================

selected = (
    selected
    .withColumn(
        "BedroomBand",
        F.when(
            F.col("Bedrooms").isNull(),
            F.lit("Unknown"),
        )
        .when(
            F.col("Bedrooms") == 0,
            F.lit("Studio/0"),
        )
        .when(
            F.col("Bedrooms") <= 2,
            F.lit("1-2"),
        )
        .when(
            F.col("Bedrooms") == 3,
            F.lit("3"),
        )
        .when(
            F.col("Bedrooms") == 4,
            F.lit("4"),
        )
        .otherwise(F.lit("5+")),
    )
    .withColumn(
        "BedroomBandSort",
        F.when(
            F.col("Bedrooms").isNull(),
            F.lit(99),
        )
        .when(
            F.col("Bedrooms") == 0,
            F.lit(0),
        )
        .when(
            F.col("Bedrooms") <= 2,
            F.lit(1),
        )
        .when(
            F.col("Bedrooms") == 3,
            F.lit(2),
        )
        .when(
            F.col("Bedrooms") == 4,
            F.lit(3),
        )
        .otherwise(F.lit(4))
        .cast(IntegerType()),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 17, Finished, Available, Finished, False)

In [16]:
# ============================================================
# 15. Add land-size bands
# ============================================================

selected = (
    selected
    .withColumn(
        "LandSizeBand",
        F.when(
            F.col("LandSizeSqm").isNull(),
            F.lit("Unknown"),
        )
        .when(
            F.col("LandSizeSqm") < 20,
            F.lit("<20"),
        )
        .when(
            F.col("LandSizeSqm") < 300,
            F.lit("20-299"),
        )
        .when(
            F.col("LandSizeSqm") < 500,
            F.lit("300-499"),
        )
        .when(
            F.col("LandSizeSqm") < 700,
            F.lit("500-699"),
        )
        .when(
            F.col("LandSizeSqm") < 1000,
            F.lit("700-999"),
        )
        .when(
            F.col("LandSizeSqm") < 2000,
            F.lit("1,000-1,999"),
        )
        .when(
            F.col("LandSizeSqm") < 10_000,
            F.lit("2,000-9,999"),
        )
        .otherwise(F.lit("10,000+")),
    )
    .withColumn(
        "LandSizeBandSort",
        F.when(
            F.col("LandSizeSqm").isNull(),
            F.lit(99),
        )
        .when(
            F.col("LandSizeSqm") < 20,
            F.lit(0),
        )
        .when(
            F.col("LandSizeSqm") < 300,
            F.lit(1),
        )
        .when(
            F.col("LandSizeSqm") < 500,
            F.lit(2),
        )
        .when(
            F.col("LandSizeSqm") < 700,
            F.lit(3),
        )
        .when(
            F.col("LandSizeSqm") < 1000,
            F.lit(4),
        )
        .when(
            F.col("LandSizeSqm") < 2000,
            F.lit(5),
        )
        .when(
            F.col("LandSizeSqm") < 10_000,
            F.lit(6),
        )
        .otherwise(F.lit(7))
        .cast(IntegerType()),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 18, Finished, Available, Finished, False)

In [17]:
# ============================================================
# 16. Add price bands
# ============================================================

selected = (
    selected
    .withColumn(
        "PriceBand100kLower",
        F.when(
            F.col("PriceValue").isNotNull(),
            (
                F.floor(
                    F.col("PriceValue") / F.lit(100_000)
                ) * F.lit(100_000)
            ).cast(LongType()),
        ).otherwise(F.lit(None).cast(LongType())),
    )
    .withColumn(
        "PriceBand100kUpper",
        F.when(
            F.col("PriceBand100kLower").isNotNull(),
            (
                F.col("PriceBand100kLower")
                + F.lit(99_999)
            ).cast(LongType()),
        ).otherwise(F.lit(None).cast(LongType())),
    )
    .withColumn(
        "PriceBand100k",
        F.when(
            F.col("PriceValue").isNull(),
            F.lit("Unknown"),
        )
        .when(
            F.col("PriceValue") >= 5_000_000,
            F.lit("$5.0m+"),
        )
        .otherwise(
            F.concat(
                F.lit("$"),
                F.format_number(
                    F.col("PriceBand100kLower"),
                    0,
                ),
                F.lit("-$"),
                F.format_number(
                    F.col("PriceBand100kUpper"),
                    0,
                ),
            )
        ),
    )
    .withColumn(
        "PriceBand100kSort",
        F.when(
            F.col("PriceValue").isNull(),
            F.lit(99_999_999),
        )
        .otherwise(
            F.col("PriceBand100kLower")
        )
        .cast(LongType()),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 19, Finished, Available, Finished, False)

In [18]:
# ============================================================
# 17. Add broader price bands
# ============================================================

selected = (
    selected
    .withColumn(
        "PriceBand",
        F.when(
            F.col("PriceValue").isNull(),
            F.lit("Unknown"),
        )
        .when(
            F.col("PriceValue") < 300_000,
            F.lit("<$300k"),
        )
        .when(
            F.col("PriceValue") < 500_000,
            F.lit("$300k-$499k"),
        )
        .when(
            F.col("PriceValue") < 750_000,
            F.lit("$500k-$749k"),
        )
        .when(
            F.col("PriceValue") < 1_000_000,
            F.lit("$750k-$999k"),
        )
        .when(
            F.col("PriceValue") < 1_500_000,
            F.lit("$1.0m-$1.49m"),
        )
        .when(
            F.col("PriceValue") < 2_000_000,
            F.lit("$1.5m-$1.99m"),
        )
        .when(
            F.col("PriceValue") < 3_000_000,
            F.lit("$2.0m-$2.99m"),
        )
        .when(
            F.col("PriceValue") < 5_000_000,
            F.lit("$3.0m-$4.99m"),
        )
        .otherwise(F.lit("$5.0m+")),
    )
    .withColumn(
        "PriceBandSort",
        F.when(
            F.col("PriceValue").isNull(),
            F.lit(99),
        )
        .when(
            F.col("PriceValue") < 300_000,
            F.lit(1),
        )
        .when(
            F.col("PriceValue") < 500_000,
            F.lit(2),
        )
        .when(
            F.col("PriceValue") < 750_000,
            F.lit(3),
        )
        .when(
            F.col("PriceValue") < 1_000_000,
            F.lit(4),
        )
        .when(
            F.col("PriceValue") < 1_500_000,
            F.lit(5),
        )
        .when(
            F.col("PriceValue") < 2_000_000,
            F.lit(6),
        )
        .when(
            F.col("PriceValue") < 3_000_000,
            F.lit(7),
        )
        .when(
            F.col("PriceValue") < 5_000_000,
            F.lit(8),
        )
        .otherwise(F.lit(9))
        .cast(IntegerType()),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 20, Finished, Available, Finished, False)

In [19]:
# ============================================================
# 18. Add analytical usability classification
# ============================================================

selected = (
    selected
    .withColumn(
        "AnalyticalRecordStatus",
        F.when(
            ~F.col("HasValidListingId"),
            F.lit("INVALID_LISTING_ID"),
        )
        .when(
            F.col("HasValidNumericPrice")
            & F.col("HasValidSoldDate")
            & F.col("HasUsableAddress"),
            F.lit("CORE_ANALYTICAL"),
        )
        .when(
            F.col("HasValidNumericPrice")
            & F.col("HasValidSoldDate"),
            F.lit("PRICE_AND_DATE"),
        )
        .when(
            F.col("HasValidSoldDate"),
            F.lit("DATE_ONLY"),
        )
        .when(
            F.col("HasValidNumericPrice"),
            F.lit("PRICE_ONLY"),
        )
        .otherwise(F.lit("LIMITED")),
    )
    .withColumn(
        "IsCoreAnalyticalRecord",
        (
            F.col("AnalyticalRecordStatus")
            == "CORE_ANALYTICAL"
        ).cast(BooleanType()),
    )
    .withColumn(
        "SelectedAt",
        F.current_timestamp(),
    )
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 21, Finished, Available, Finished, False)

In [20]:
# ============================================================
# 19. Select final analytical schema
# ============================================================

analysis = selected.select(
    # Listing grain and selected source observation
    "ListingId",
    "SelectedObservationKey",
    "SelectedRecordHash",

    # Selection explanation
    "SelectionScore",
    "SelectionAttributes",
    "SelectionRank",
    "SelectedAt",

    # Observation history
    "ObservationCount",
    "DistinctRecordVersionCount",
    "FirstScrapedAt",
    "LastScrapedAt",
    "FirstScrapeDate",
    "LastScrapeDate",
    "DistinctNumericPriceCount",
    "DistinctAddressCount",
    "DistinctSoldDateCount",
    "DistinctPropertyTypeCount",

    # Variation flags
    "HasMultipleObservations",
    "HasMultipleRecordVersions",
    "HasPriceVariation",
    "HasAddressVariation",
    "HasSoldDateVariation",
    "HasPropertyTypeVariation",

    # Whether any observation possessed useful data
    "AnyObservationHasValidNumericPrice",
    "AnyObservationHasValidSoldDate",
    "AnyObservationHasUsableAddress",
    "AnyObservationHasValidLandSize",

    # Selected sale date
    "SoldDate",
    "SoldYear",
    "SoldMonth",
    "SoldMonthName",
    "SoldMonthShortName",
    "SoldQuarter",
    "SoldQuarterLabel",
    "SoldYearMonth",
    "SoldYearMonthSort",

    # Geography
    "Suburb",
    "Postcode",
    "FullAddress",
    "IsAddressSuppressed",

    # Property attributes
    "PropertyType",
    "Bedrooms",
    "BedroomBand",
    "BedroomBandSort",
    "Bathrooms",
    "CarSpaces",
    "LandSizeSqm",
    "LandSizeBand",
    "LandSizeBandSort",

    # Price attributes
    "PriceText",
    "PriceValue",
    "PriceBand100k",
    "PriceBand100kSort",
    "PriceBand100kLower",
    "PriceBand100kUpper",
    "PriceBand",
    "PriceBandSort",
    "PricePerSqm",
    "PriceType",
    "HasNumericPrice",
    "IsContactAgent",
    "IsPriceRange",
    "IsPriceFrom",
    "IsPriceWithheld",

    # Selected observation source
    "DetailPath",
    "DetailUrl",
    "ScrapedAt",
    "ScrapeDate",
    "SourceFile",
    "SourceUrl",
    "SourcePage",
    "SourceOrdinal",

    # Source repair metadata
    "InputWasColumnShifted",

    # Quality classifications
    "ListingIdQuality",
    "SoldDateQuality",
    "PriceQuality",
    "AddressQuality",
    "LandSizeQuality",
    "ScrapeTimestampQuality",

    # Quality flags
    "HasValidListingId",
    "HasValidSoldDate",
    "HasValidNumericPrice",
    "HasUsableAddress",
    "HasValidLandSize",

    # Overall analytical usability
    "AnalyticalRecordStatus",
    "IsCoreAnalyticalRecord",
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 22, Finished, Available, Finished, False)

In [21]:
# ============================================================
# 20. Validate analytical grain before write
# ============================================================

analysis_validation = analysis.select(
    F.count("*").alias("RowCount"),
    F.countDistinct("ListingId").alias(
        "DistinctListingCount"
    ),
    F.countDistinct("SelectedObservationKey").alias(
        "DistinctSelectedObservationCount"
    ),
    F.sum(
        F.when(
            F.col("ListingId").isNull(),
            1,
        ).otherwise(0)
    ).alias("NullListingIdCount"),
    F.sum(
        F.when(
            F.col("SelectedObservationKey").isNull(),
            1,
        ).otherwise(0)
    ).alias("NullSelectedObservationKeyCount"),
).first()

if analysis_validation["NullListingIdCount"] != 0:
    raise RuntimeError(
        "Analytical output contains NULL ListingId values."
    )

if (
    analysis_validation[
        "NullSelectedObservationKeyCount"
    ]
    != 0
):
    raise RuntimeError(
        "Analytical output contains NULL "
        "SelectedObservationKey values."
    )

if (
    analysis_validation["RowCount"]
    != analysis_validation["DistinctListingCount"]
):
    raise RuntimeError(
        "Analytical output is not unique by ListingId."
    )

if (
    analysis_validation["RowCount"]
    != source_listing_count
):
    raise RuntimeError(
        "Analytical output row count does not equal the "
        "distinct listing count in Cleaned. "
        f"Expected={source_listing_count:,}; "
        f"Actual={analysis_validation['RowCount']:,}"
    )


# ============================================================
# 21. Publish analytical table
# ============================================================

(
    analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ANALYSIS_TABLE)
)

print(
    f"Wrote analytical table: {ANALYSIS_TABLE}"
)


# ============================================================
# 22. Validate published tables
# ============================================================

observation_summary = spark.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT ObservationKey)
            AS distinct_observation_count,
        COUNT(DISTINCT ListingId)
            AS distinct_listing_count,
        MIN(ScrapedAt) AS min_scraped_at,
        MAX(ScrapedAt) AS max_scraped_at

    FROM {OBSERVATION_TABLE}
    """
)

analysis_summary = spark.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT ListingId)
            AS distinct_listing_count,
        COUNT(DISTINCT SelectedObservationKey)
            AS distinct_selected_observation_count,

        SUM(
            CASE
                WHEN PriceValue IS NULL THEN 1
                ELSE 0
            END
        ) AS null_price_rows,

        SUM(
            CASE
                WHEN SoldDate IS NULL THEN 1
                ELSE 0
            END
        ) AS null_sold_date_rows,

        SUM(
            CASE
                WHEN FullAddress IS NULL THEN 1
                ELSE 0
            END
        ) AS null_address_rows,

        SUM(
            CASE
                WHEN HasMultipleObservations THEN 1
                ELSE 0
            END
        ) AS multiple_observation_listings,

        SUM(
            CASE
                WHEN HasMultipleRecordVersions THEN 1
                ELSE 0
            END
        ) AS multiple_version_listings,

        SUM(
            CASE
                WHEN HasPriceVariation THEN 1
                ELSE 0
            END
        ) AS price_variation_listings,

        SUM(
            CASE
                WHEN IsCoreAnalyticalRecord THEN 1
                ELSE 0
            END
        ) AS core_analytical_rows,

        MIN(SoldDate) AS min_sold_date,
        MAX(SoldDate) AS max_sold_date,

        COUNT(DISTINCT Suburb) AS suburb_count

    FROM {ANALYSIS_TABLE}
    """
)

print("\nCurated observation table:")
observation_summary.show(truncate=False)

print("\nCurated analytical table:")
analysis_summary.show(truncate=False)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 23, Finished, Available, Finished, False)

Wrote analytical table: Curated.property_sales_analysis

Curated observation table:
+---------+--------------------------+----------------------+-----------------------+-----------------------+
|row_count|distinct_observation_count|distinct_listing_count|min_scraped_at         |max_scraped_at         |
+---------+--------------------------+----------------------+-----------------------+-----------------------+
|347887   |347887                    |296422                |2026-07-02 13:06:09.289|2026-07-09 16:32:55.348|
+---------+--------------------------+----------------------+-----------------------+-----------------------+


Curated analytical table:
+---------+----------------------+-----------------------------------+---------------+-------------------+-----------------+-----------------------------+-------------------------+------------------------+--------------------+-------------+-------------+------------+
|row_count|distinct_listing_count|distinct_selected_observation_count|

In [22]:
# ============================================================
# 23. Validate published row counts
# ============================================================

published_observation = observation_summary.first()
published_analysis = analysis_summary.first()

if published_observation["row_count"] != source_row_count:
    raise RuntimeError(
        "Published observation row count differs from Cleaned."
    )

if (
    published_observation["row_count"]
    != published_observation[
        "distinct_observation_count"
    ]
):
    raise RuntimeError(
        "Published observation table is not unique by "
        "ObservationKey."
    )

if (
    published_analysis["row_count"]
    != published_analysis["distinct_listing_count"]
):
    raise RuntimeError(
        "Published analytical table is not unique by ListingId."
    )

if published_analysis["row_count"] != source_listing_count:
    raise RuntimeError(
        "Published analytical row count differs from the "
        "Cleaned distinct listing count."
    )

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 24, Finished, Available, Finished, False)

In [23]:
# ============================================================
# 24. Profile observation counts
# ============================================================

print("\nListing observation-count distribution:")

spark.sql(
    f"""
    SELECT
        CASE
            WHEN ObservationCount = 1 THEN '1'
            WHEN ObservationCount = 2 THEN '2'
            WHEN ObservationCount = 3 THEN '3'
            WHEN ObservationCount BETWEEN 4 AND 5 THEN '4-5'
            WHEN ObservationCount BETWEEN 6 AND 10 THEN '6-10'
            ELSE '11+'
        END AS ObservationCountBand,

        CASE
            WHEN ObservationCount = 1 THEN 1
            WHEN ObservationCount = 2 THEN 2
            WHEN ObservationCount = 3 THEN 3
            WHEN ObservationCount BETWEEN 4 AND 5 THEN 4
            WHEN ObservationCount BETWEEN 6 AND 10 THEN 5
            ELSE 6
        END AS ObservationCountBandSort,

        COUNT(*) AS ListingCount,

        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (),
            2
        ) AS Percentage

    FROM {ANALYSIS_TABLE}

    GROUP BY
        CASE
            WHEN ObservationCount = 1 THEN '1'
            WHEN ObservationCount = 2 THEN '2'
            WHEN ObservationCount = 3 THEN '3'
            WHEN ObservationCount BETWEEN 4 AND 5 THEN '4-5'
            WHEN ObservationCount BETWEEN 6 AND 10 THEN '6-10'
            ELSE '11+'
        END,
        CASE
            WHEN ObservationCount = 1 THEN 1
            WHEN ObservationCount = 2 THEN 2
            WHEN ObservationCount = 3 THEN 3
            WHEN ObservationCount BETWEEN 4 AND 5 THEN 4
            WHEN ObservationCount BETWEEN 6 AND 10 THEN 5
            ELSE 6
        END

    ORDER BY ObservationCountBandSort
    """
).show(truncate=False)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 25, Finished, Available, Finished, False)


Listing observation-count distribution:
+--------------------+------------------------+------------+----------+
|ObservationCountBand|ObservationCountBandSort|ListingCount|Percentage|
+--------------------+------------------------+------------+----------+
|1                   |1                       |265912      |89.71     |
|2                   |2                       |21922       |7.40      |
|3                   |3                       |4005        |1.35      |
|4-5                 |4                       |2500        |0.84      |
|6-10                |5                       |2044        |0.69      |
|11+                 |6                       |39          |0.01      |
+--------------------+------------------------+------------+----------+



In [24]:
# ============================================================
# 25. Profile analytical status
# ============================================================

print("\nAnalytical record status:")

spark.sql(
    f"""
    SELECT
        AnalyticalRecordStatus,
        COUNT(*) AS ListingCount,

        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (),
            2
        ) AS Percentage

    FROM {ANALYSIS_TABLE}

    GROUP BY AnalyticalRecordStatus

    ORDER BY
        ListingCount DESC,
        AnalyticalRecordStatus
    """
).show(truncate=False)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 26, Finished, Available, Finished, False)


Analytical record status:
+----------------------+------------+----------+
|AnalyticalRecordStatus|ListingCount|Percentage|
+----------------------+------------+----------+
|CORE_ANALYTICAL       |232596      |78.47     |
|DATE_ONLY             |57182       |19.29     |
|LIMITED               |3679        |1.24      |
|PRICE_AND_DATE        |2778        |0.94      |
|PRICE_ONLY            |187         |0.06      |
+----------------------+------------+----------+



In [25]:
# ============================================================
# 26. Profile selected quality
# ============================================================

quality_dimensions = [
    "SoldDateQuality",
    "PriceQuality",
    "AddressQuality",
    "LandSizeQuality",
]

for quality_column in quality_dimensions:
    print(f"\nSelected {quality_column}:")

    spark.sql(
        f"""
        SELECT
            {quality_column},
            COUNT(*) AS ListingCount,

            ROUND(
                COUNT(*) * 100.0
                / SUM(COUNT(*)) OVER (),
                2
            ) AS Percentage

        FROM {ANALYSIS_TABLE}

        GROUP BY {quality_column}

        ORDER BY
            ListingCount DESC,
            {quality_column}
        """
    ).show(truncate=False)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 27, Finished, Available, Finished, False)


Selected SoldDateQuality:
+------------------+------------+----------+
|SoldDateQuality   |ListingCount|Percentage|
+------------------+------------+----------+
|VALID             |292556      |98.70     |
|MISSING           |3850        |1.30      |
|HISTORICAL_OUTLIER|16          |0.01      |
+------------------+------------+----------+


Selected PriceQuality:
+-------------------+------------+----------+
|PriceQuality       |ListingCount|Percentage|
+-------------------+------------+----------+
|VALID_NUMERIC      |235561      |79.47     |
|CONTACT_AGENT      |58344       |19.68     |
|RANGE_NUMERIC_VALUE|2394        |0.81      |
|IMPLAUSIBLY_LOW    |99          |0.03      |
|IMPLAUSIBLY_HIGH   |24          |0.01      |
+-------------------+------------+----------+


Selected AddressQuality:
+--------------+------------+----------+
|AddressQuality|ListingCount|Percentage|
+--------------+------------+----------+
|VALID         |291244      |98.25     |
|SUPPRESSED    |5178        

In [26]:
# ============================================================
# 27. Inspect listings with meaningful source variation
# ============================================================

print("\nListings with observation variation:")

spark.sql(
    f"""
    SELECT
        ListingId,
        ObservationCount,
        DistinctRecordVersionCount,
        DistinctNumericPriceCount,
        DistinctAddressCount,
        DistinctSoldDateCount,
        HasPriceVariation,
        HasAddressVariation,
        HasSoldDateVariation,
        FirstScrapedAt,
        LastScrapedAt,
        PriceValue,
        SoldDate,
        FullAddress,
        SelectedObservationKey,
        SelectionScore,
        SelectionAttributes

    FROM {ANALYSIS_TABLE}

    WHERE
        HasMultipleRecordVersions = true
        OR HasPriceVariation = true
        OR HasAddressVariation = true
        OR HasSoldDateVariation = true

    ORDER BY
        ObservationCount DESC,
        DistinctRecordVersionCount DESC,
        ListingId

    LIMIT 100
    """
).show(truncate=False)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 28, Finished, Available, Finished, False)


Listings with observation variation:
+---------+----------------+--------------------------+-------------------------+--------------------+---------------------+-----------------+-------------------+--------------------+-----------------------+-----------------------+----------+----------+---------------------------+----------------------------------------------------------------+--------------+------------------------------------------------------------------------------------------------------+
|ListingId|ObservationCount|DistinctRecordVersionCount|DistinctNumericPriceCount|DistinctAddressCount|DistinctSoldDateCount|HasPriceVariation|HasAddressVariation|HasSoldDateVariation|FirstScrapedAt         |LastScrapedAt          |PriceValue|SoldDate  |FullAddress                |SelectedObservationKey                                          |SelectionScore|SelectionAttributes                                                                                   |
+---------+----------------+----

In [27]:
# ============================================================
# 28. Inspect historical sold-date outliers
# ============================================================

print("\nHistorical sold-date outliers:")

spark.sql(
    f"""
    SELECT
        ListingId,
        SoldDate,
        SoldDateQuality,
        PriceValue,
        FullAddress,
        Suburb,
        PropertyType,
        ScrapedAt,
        DetailUrl,
        SelectedObservationKey

    FROM {ANALYSIS_TABLE}

    WHERE SoldDateQuality = 'HISTORICAL_OUTLIER'

    ORDER BY
        SoldDate,
        ListingId

    LIMIT 100
    """
).show(truncate=False)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 29, Finished, Available, Finished, False)


Historical sold-date outliers:
+---------+----------+------------------+----------+---------------------------------+---------------+------------+-----------------------+------------------------------------------------------------------------------+----------------------------------------------------------------+
|ListingId|SoldDate  |SoldDateQuality   |PriceValue|FullAddress                      |Suburb         |PropertyType|ScrapedAt              |DetailUrl                                                                     |SelectedObservationKey                                          |
+---------+----------+------------------+----------+---------------------------------+---------------+------------+-----------------------+------------------------------------------------------------------------------+----------------------------------------------------------------+
|110117305|1901-12-14|HISTORICAL_OUTLIER|NULL      |42 Circe Circle, Dalkeith        |Dalkeith       |House       |2

In [28]:
# ============================================================
# 29. Final status
# ============================================================

print(
    f"""
Curated property-sales publication complete.

Source table:
    {SOURCE_TABLE}

Observation-history table:
    {OBSERVATION_TABLE}

Analytical table:
    {ANALYSIS_TABLE}

Source observations:                 {source_row_count:,}
Published observations:              {published_observation["row_count"]:,}
Distinct source listings:            {source_listing_count:,}
Published analytical listings:       {published_analysis["row_count"]:,}

Listings without numeric price:      {published_analysis["null_price_rows"]:,}
Listings without sold date:          {published_analysis["null_sold_date_rows"]:,}
Listings without usable address:     {published_analysis["null_address_rows"]:,}

Listings with multiple observations: {published_analysis["multiple_observation_listings"]:,}
Listings with multiple versions:     {published_analysis["multiple_version_listings"]:,}
Listings with price variation:       {published_analysis["price_variation_listings"]:,}

Core analytical records:             {published_analysis["core_analytical_rows"]:,}

Minimum selected sold date:          {published_analysis["min_sold_date"]}
Maximum selected sold date:          {published_analysis["max_sold_date"]}
Distinct selected suburbs:           {published_analysis["suburb_count"]:,}
"""
)

StatementMeta(, 96eb78ff-cd6f-4700-8157-49c7bf024b53, 30, Finished, Available, Finished, False)


Curated property-sales publication complete.

Source table:
    Cleaned.property_sales_data

Observation-history table:
    Curated.property_sales_observation

Analytical table:
    Curated.property_sales_analysis

Source observations:                 347,887
Published observations:              347,887
Distinct source listings:            296,422
Published analytical listings:       296,422

Listings without numeric price:      58,344
Listings without sold date:          3,850
Listings without usable address:     5,178

Listings with multiple observations: 30,510
Listings with multiple versions:     30,510
Listings with price variation:       0

Core analytical records:             232,596

Minimum selected sold date:          1901-12-14
Maximum selected sold date:          2026-07-09
Distinct selected suburbs:           333

